# Câu 2 - Tích chập, Lọc trung vị, Ngưỡng ảnh

Cho ảnh màu I, chuyển sang ảnh xám, sau đó:
- Tích chập với kernel 3x3 (pad=1) → **I1**
- Tích chập với kernel 5x5 (pad=2) → **I2**
- Tích chập với kernel 7x7 (pad=3, stride=2) → **I3**
- Lọc trung vị I3 với neighbors 3x3 → **I4**
- Lọc trung bình I1 với neighbors 5x5 → **I5**
- Lấy ngưỡng so sánh I4, I5 → **I6**

## 0. Import thư viện

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

## 1. Đọc ảnh màu và chuyển sang ảnh xám

Công thức chuyển xám (OpenCV đọc ảnh theo thứ tự kênh **BGR**):

$$Gray = 0.299 \times R + 0.587 \times G + 0.114 \times B$$

In [ ]:
file_name = "anh1.jpg"
img = cv2.imread(file_name)

if img is None:
    raise FileNotFoundError("Không tìm thấy ảnh")

h, w, c = img.shape

gray = np.zeros((h, w), dtype=np.uint8)

for i in range(h):
    for j in range(w):
        B = img[i, j, 0]
        G = img[i, j, 1]
        R = img[i, j, 2]

        gray_value = 0.299 * R + 0.587 * G + 0.114 * B
        gray[i, j] = int(round(gray_value))

print("Kích thước ảnh gốc:", img.shape)
print("Kích thước ảnh xám:", gray.shape)

plt.imshow(gray, cmap="gray")
plt.title("Ảnh xám")
plt.axis("off")
plt.show()

## 2. Hàm thêm Padding (viền 0 quanh ảnh)

**Mục đích:** khi kernel trượt tới sát biên ảnh, nó vẫn có đủ pixel xung quanh để tính toán, không bị tràn ra ngoài mảng.

`pad` = số lớp viền 0 thêm vào mỗi cạnh.

In [ ]:
def add_padding(image, pad):
    old_h, old_w = image.shape
    new_h = old_h + 2 * pad
    new_w = old_w + 2 * pad

    padded = np.zeros((new_h, new_w), dtype=np.float64)

    # copy ảnh gốc vào giữa, phần viền xung quanh giữ giá trị 0
    for i in range(old_h):
        for j in range(old_w):
            padded[i + pad, j + pad] = image[i, j]

    return padded

### Thử nghiệm hàm `add_padding` trên 1 ma trận nhỏ để kiểm tra

In [ ]:
test_matrix = np.array([[1, 2], [3, 4]])
test_padded = add_padding(test_matrix, pad=1)
print("Ma trận gốc:")
print(test_matrix)
print("\nMa trận sau khi thêm padding=1:")
print(test_padded)

## 3. Hàm phép tích chập (Convolution)

Tham số:
- `image`: ảnh xám đầu vào (2 chiều)
- `kernel`: ma trận trọng số (vd 3x3, 5x5, 7x7)
- `pad`: số lớp padding thêm vào ảnh
- `stride`: bước nhảy khi trượt kernel

Công thức tích chập tại 1 điểm (x, y):

$$output(x,y) = \sum_{a,b} kernel(a,b) \times image(x+a, y+b)$$

Công thức kích thước ảnh output:

$$out\_size = \frac{image\_size + 2 \times pad - kernel\_size}{stride} + 1$$

In [ ]:
def convolution(image, kernel, pad, stride=1, clip_output=True):
    img_h, img_w = image.shape
    k_h, k_w = kernel.shape

    # Bước 1: thêm padding cho ảnh gốc
    padded = add_padding(image, pad)

    # Bước 2: tính kích thước ảnh output theo công thức chuẩn
    out_h = (img_h + 2 * pad - k_h) // stride + 1
    out_w = (img_w + 2 * pad - k_w) // stride + 1

    output = np.zeros((out_h, out_w), dtype=np.float64)

    # Bước 3: trượt kernel khắp ảnh đã padding
    for out_i in range(out_h):
        for out_j in range(out_w):
            # vị trí bắt đầu (góc trên-trái) của vùng ảnh đang xét
            start_i = out_i * stride
            start_j = out_j * stride

            # lấy ra vùng ảnh con có cùng kích thước với kernel
            region = padded[start_i:start_i + k_h, start_j:start_j + k_w]

            # nhân từng phần tử với kernel rồi cộng lại (tích chập)
            value = 0.0
            for a in range(k_h):
                for b in range(k_w):
                    value += kernel[a, b] * region[a, b]

            output[out_i, out_j] = value

    # Bước 4: ép giá trị về khoảng [0, 255] và kiểu uint8 (ảnh hợp lệ)
    if clip_output:
        output = np.clip(output, 0, 255).astype(np.uint8)

    return output

## 4. Tạo các kernel Box blur (trung bình cộng)

Box blur: mỗi phần tử kernel = $\frac{1}{kích\_thước \times kích\_thước}$

Ý nghĩa: giá trị pixel mới = trung bình cộng các pixel lân cận → làm mờ ảnh, giảm nhiễu.

In [ ]:
sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float64)
sobel_y = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=np.float64)

kernel_5x5 = np.array([[1, 4, 6, 4, 1], [4, 16, 24, 16, 4],
                         [6, 24, 36, 24, 6], [4, 16, 24, 16, 4],
                         [1, 4, 6, 4, 1]], dtype=np.float64) / 256.0
g7 = np.array([1, 6, 15, 20, 15, 6, 1], dtype=np.float64)
kernel_7x7 = np.outer(g7, g7) / 4096.0
mean_5x5 = np.ones((5, 5), dtype=np.float64) / 25.0

print("Sobel X 3x3:")
print(sobel_x)

## 5. Tính I1: Sobel 3x3 phát hiện cạnh, padding = 1

In [ ]:
Gx = convolution(gray, sobel_x, pad=1, stride=1, clip_output=False)
Gy = convolution(gray, sobel_y, pad=1, stride=1, clip_output=False)
I1 = np.clip(np.hypot(Gx, Gy), 0, 255).astype(np.uint8)
print("Kích thước I1:", I1.shape)

plt.imshow(I1, cmap="gray")
plt.title("I1 - Sobel 3x3 (phát hiện cạnh)")
plt.axis("off")
plt.show()

## 6. Tính I2: Gaussian 5x5 làm mờ, padding = 2

In [ ]:
I2 = convolution(gray, kernel_5x5, pad=2, stride=1)
print("Kích thước I2:", I2.shape)

plt.imshow(I2, cmap="gray")
plt.title("I2 - Gaussian 5x5 (làm mờ)")
plt.axis("off")
plt.show()

## 7. Tính I3: Gaussian 7x7, padding = 3, stride = 2

Lưu ý: vì `stride = 2` nên ảnh output **I3 sẽ nhỏ hơn** I1, I2 (kích thước giảm gần một nửa).

In [ ]:
I3 = convolution(gray, kernel_7x7, pad=3, stride=2)
print("Kích thước I3:", I3.shape)
print("(I3 nhỏ hơn I1, I2 vì stride=2 làm ảnh output bị thu nhỏ một nửa)")

plt.imshow(I3, cmap="gray")
plt.title("I3 - Gaussian 7x7, stride=2 (thu nhỏ)")
plt.axis("off")
plt.show()

## 8. Hàm lọc trung vị (Median Filter)

**Ý tưởng:** tại mỗi pixel, lấy tất cả pixel trong vùng lân cận (vd 3x3 hoặc 5x5), **sắp xếp tăng dần**, rồi lấy giá trị **ở giữa** (trung vị / median) làm giá trị pixel mới.

**Khác Box blur:** median filter không nhân-cộng theo trọng số mà sắp xếp rồi chọn giá trị giữa → loại nhiễu muối-tiêu (salt-pepper) tốt hơn, giữ cạnh rõ hơn.

In [ ]:
def median_filter(image, ksize):
    img_h, img_w = image.shape
    pad = ksize // 2  # số lớp padding cần thêm để vùng lân cận đủ pixel

    padded = add_padding(image, pad)

    output = np.zeros((img_h, img_w), dtype=np.uint8)

    for i in range(img_h):
        for j in range(img_w):
            # lấy vùng lân cận ksize x ksize quanh pixel (i, j)
            region = padded[i:i + ksize, j:j + ksize]

            # chuyển vùng 2D thành danh sách 1D rồi sắp xếp tăng dần
            values = []
            for a in range(ksize):
                for b in range(ksize):
                    values.append(region[a, b])
            values.sort()

            # lấy giá trị ở chính giữa danh sách đã sắp xếp = trung vị
            median_value = values[len(values) // 2]

            output[i, j] = int(median_value)

    return output

## 9. Tính I4: lọc trung vị ảnh I3 với neighbors 3x3

In [ ]:
I4 = median_filter(I3, ksize=3)
print("Kích thước I4:", I4.shape)

plt.imshow(I4, cmap="gray")
plt.title("I4 (median filter của I3, 3x3)")
plt.axis("off")
plt.show()

## 10. Tính I5: lọc trung bình (Mean Filter) ảnh I1 với neighbors 5x5

In [ ]:
I5 = convolution(I1, mean_5x5, pad=2, stride=1)
print("Kích thước I5:", I5.shape)

plt.imshow(I5, cmap="gray")
plt.title("I5 - Mean Filter 5x5 trên I1")
plt.axis("off")
plt.show()

## 11. Hàm đưa 2 ảnh về cùng kích thước

Vì I4 (suy ra từ I3 có stride=2) sẽ **nhỏ hơn** I5 (suy ra từ I1, stride=1), nên trước khi so sánh pixel-by-pixel, ta cần đưa ảnh nhỏ hơn lên cùng kích thước ảnh lớn hơn bằng cách thêm viền 0 và đặt ảnh gốc vào **chính giữa**.

In [ ]:
def make_same_size(imageA, imageB):
    hA, wA = imageA.shape
    hB, wB = imageB.shape

    if (hA, wA) == (hB, wB):
        # đã cùng kích thước, không cần làm gì
        return imageA, imageB

    # xác định kích thước lớn nhất giữa 2 ảnh
    target_h = max(hA, hB)
    target_w = max(wA, wB)

    # hàm phụ: đặt ảnh nhỏ vào giữa 1 nền 0 có kích thước target
    def pad_to_target(image, target_h, target_w):
        old_h, old_w = image.shape
        new_img = np.zeros((target_h, target_w), dtype=np.uint8)

        # tính lề trên/trái để đặt ảnh gốc vào CHÍNH GIỮA nền mới
        offset_h = (target_h - old_h) // 2
        offset_w = (target_w - old_w) // 2

        for i in range(old_h):
            for j in range(old_w):
                new_img[i + offset_h, j + offset_w] = image[i, j]

        return new_img

    imageA_new = pad_to_target(imageA, target_h, target_w)
    imageB_new = pad_to_target(imageB, target_h, target_w)

    return imageA_new, imageB_new

## 12. Xây dựng I6 bằng phép lấy ngưỡng

Quy tắc:
- Nếu $I4(x,y) > I5(x,y)$ → $I6(x,y) = 0$
- Ngược lại → $I6(x,y) = I5(x,y)$

Trước tiên phải đưa I4, I5 về cùng kích thước bằng hàm `make_same_size` ở trên.

In [ ]:
print("Kích thước I4 trước khi chỉnh:", I4.shape)
print("Kích thước I5 trước khi chỉnh:", I5.shape)

I4_resized, I5_resized = make_same_size(I4, I5)

print("Kích thước I4 sau khi chỉnh:", I4_resized.shape)
print("Kích thước I5 sau khi chỉnh:", I5_resized.shape)

new_h, new_w = I4_resized.shape
I6 = np.zeros((new_h, new_w), dtype=np.uint8)

for i in range(new_h):
    for j in range(new_w):
        if I4_resized[i, j] > I5_resized[i, j]:
            I6[i, j] = 0
        else:
            I6[i, j] = I5_resized[i, j]

print("Kích thước I6:", I6.shape)

plt.imshow(I6, cmap="gray")
plt.title("I6 (ngưỡng: I4 > I5 thì = 0, ngược lại = I5)")
plt.axis("off")
plt.show()

## 13. Hiển thị tổng hợp tất cả kết quả

In [ ]:
plt.figure(figsize=(15, 8))

plt.subplot(2, 4, 1)
plt.imshow(gray, cmap="gray")
plt.title("Ảnh xám gốc")
plt.axis("off")

plt.subplot(2, 4, 2)
plt.imshow(I1, cmap="gray")
plt.title("I1 - Sobel 3x3")
plt.axis("off")

plt.subplot(2, 4, 3)
plt.imshow(I2, cmap="gray")
plt.title("I2 - Gaussian 5x5")
plt.axis("off")

plt.subplot(2, 4, 4)
plt.imshow(I3, cmap="gray")
plt.title("I3 - Gaussian 7x7, stride=2")
plt.axis("off")

plt.subplot(2, 4, 5)
plt.imshow(I4, cmap="gray")
plt.title("I4 (median I3, 3x3)")
plt.axis("off")

plt.subplot(2, 4, 6)
plt.imshow(I5, cmap="gray")
plt.title("I5 - Mean Filter 5x5 trên I1")
plt.axis("off")

plt.subplot(2, 4, 7)
plt.imshow(I6, cmap="gray")
plt.title("I6 (ngưỡng I4 vs I5)")
plt.axis("off")

plt.tight_layout()
output_dir = "output_cau2"
os.makedirs(output_dir, exist_ok=True)
plt.savefig(os.path.join(output_dir, "ket_qua_cau2.png"), dpi=100)
print(f"Đã lưu hình tổng hợp vào {output_dir}/ket_qua_cau2.png")
plt.show()

## 14. In thông tin kiểm tra (min/max/kích thước) tất cả ảnh

In [ ]:
print("====== THÔNG TIN KIỂM TRA ======")
print("Ảnh xám gốc      :", gray.shape, "min =", gray.min(), "max =", gray.max())
print("I1 (Sobel 3x3)    :", I1.shape, "min =", I1.min(), "max =", I1.max())
print("I2 (Gaussian 5x5) :", I2.shape, "min =", I2.min(), "max =", I2.max())
print("I3 (Gaussian 7x7, stride=2):", I3.shape, "min =", I3.min(), "max =", I3.max())
print("I4 (median I3, 3x3):", I4.shape, "min =", I4.min(), "max =", I4.max())
print("I5 (mean I1, 5x5):", I5.shape, "min =", I5.min(), "max =", I5.max())
print("I6 (ngưỡng)       :", I6.shape, "min =", I6.min(), "max =", I6.max())

## 15. Xuất và hiển thị riêng từng ảnh kết quả

In [ ]:
individual_results = [
    ("hinh_05_anh_xam.png", gray, "Hình 5: Ảnh xám gốc"),
    ("hinh_06_I1_sobel_3x3.png", I1, "Hình 6: I1 - Sobel 3x3 (phát hiện cạnh)"),
    ("hinh_07_I2_gaussian_5x5.png", I2, "Hình 7: I2 - Gaussian 5x5 (làm mờ)"),
    ("hinh_08_I3_gaussian_7x7_stride2.png", I3, "Hình 8: I3 - Gaussian 7x7, stride=2 (thu nhỏ)"),
    ("hinh_09_I4_median_3x3.png", I4, "Hình 9: I4 - Median Filter 3x3 trên I3"),
    ("hinh_10_I5_mean_5x5.png", I5, "Hình 10: I5 - Mean Filter 5x5 trên I1"),
    ("hinh_11_I6_ket_hop.png", I6, "Hình 11: I6 - Kết hợp I4 và I5")
]

os.makedirs(output_dir, exist_ok=True)
for file_name, result_image, caption in individual_results:
    file_path = os.path.join(output_dir, file_name)
    cv2.imwrite(file_path, result_image)
    print(f"Đã lưu {file_path}: kích thước {result_image.shape}")
    plt.figure(figsize=(8, 8))
    plt.imshow(result_image, cmap="gray", vmin=0, vmax=255)
    plt.title(caption)
    plt.axis("off")
    plt.tight_layout()
    plt.show()